# Supplementary Code to Report

Here we list all the dependencies to run each code chunk:

In [ ]:
%pip install csv json pathlib ultralytics rfdetr cv2 numpy matplotlib.pyplot sklearn.metrics torch pandas

## Section 1 - Dog Detection and Cropping

### Section 1.1 - Training and Validation Data Extraction

DO NOT RUN ANY SCRIPT! Notice - training data downloads 5000 training images and 1000 validation images locally. Validation images can be found on GitHub, training cannot as the folder size is too large. 

### Section 1.2 - Model Training

DO NOT RUN ANY MODELS - on RTX 2050, finetuning YOLO26n took 3 hours and RF-DETR-Nano took 11 hours, all models are saved as `.pth` or `.pt` files and can be run safely on a computer, demonstrated in Section 1.3. YOLO26n files initialise very quickly, RF-DETR-Nano files take slightly longer to warm up but once the model is loaded it runs fine.  

### Section 1.3 - Validation

All chunks can be run, validation images are stored on GitHub. 

In [ ]:
import csv
import json
from pathlib import Path
from ultralytics import YOLO
from rfdetr import RFDETRNano
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


#### 1.3.1 Configuration

Here we initialise the family of model (YOLO or RF-DETR), the type of model (basline or finetuned), and then the filepath based on which `.pth` or `.pt` file we are running for our validation. Options are listed in comment below, varying the inputs will allow access to different validations. We set a minimum threhold confidence to allow most predictions to pass, as well as a set IoU (Intersection over Union) threshold for box similarity, and over the course of our validation we will loop over image confidence thresholds in $[0.05, 0.95]$ across $0.05$ gaps. Image size is standardised for both models at `640px` since both models were trained on this fixed size.  

In [ ]:
MODEL_FAMILY = "YOLO"  # "YOLO" or "RFDETR" (output directory prefix)
MODEL_TYPE = "finetuned"  # "baseline" or "finetuned"
MODEL_PATH = None  # None selects default pretrained weights for baseline only.
# YOLO baseline:  MODEL_FAMILY="YOLO"; MODEL_TYPE="baseline"; MODEL_PATH="yolo26n.pt"
# YOLO finetuned: MODEL_FAMILY="YOLO"; MODEL_TYPE="finetuned";
MODEL_PATH="runs/detect/yolo26n_dog-3/weights/best.pt"
# RFDETR baseline: MODEL_FAMILY="RFDETR"; MODEL_TYPE="baseline"; MODEL_PATH=None
# RFDETR finetuned: MODEL_FAMILY="RFDETR"; MODEL_TYPE="finetuned";
#MODEL_PATH="runs/rfdetr_nano_dog_only/checkpoint_best_total.pth"
# A compatible last.ckpt may also be supplied on RF-DETR versions supporting it.
DOG_CLASS_ID_OVERRIDE = None  # Set only if the checkpoint's prediction ID is verified.
VAL_IMAGE_DIR = Path("DogCOCO/images/val")
VAL_LABEL_DIR = Path("DogCOCO/labels/val")
MIN_PRED_CONF = 0.001
IOU_MATCH_THRESHOLD = 0.50
THRESHOLDS = np.arange(0.05, 1.00, 0.05)  # exact original grid
MISSING_LABELS_ARE_NEGATIVE = False
YOLO_IMAGE_SIZE = 640
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

#### 1.3.2 - Directory and CSV Management

Here we define 4 file manipulation functions, in order to read and write our results to CSVs from which to create plots. `save_csv` inputs a collection of dictionaries and writes them to CSVs; `save_image` saves the cropped image whose coordinates are outputted by the trained models; `get_image_paths` finds where the validation images are located and allows them to be inputted to model, and `prepare_output_directory` creates the file hierarchy for each model when performing validation, written as: 
```
alt/
└── YOLO_results/
    └── validation_analysis/
        └── finetuned/
            ├── dog_crops/
            └── box_diagnostics/
```

In [ ]:
def save_csv(path, rows, fieldnames=None):
    """Retain the original CSV field order, including empty diagnostic tables."""
    if fieldnames is None:
        fieldnames = list(rows[0].keys())
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def save_image(path, image):
    if not cv2.imwrite(str(path), image):
        raise IOError(f"Could not save {path}")


def get_image_paths(image_dir):
    paths = sorted(p for p in image_dir.iterdir()
                   if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    if not paths:
        raise ValueError(f"No validation images found in {image_dir}")
    return paths


def prepare_output_directory(model_family, model_type):
    output_dir = Path("alt") / f"{model_family}_results" / "validation_analysis" / model_type
    # Avoid mixing old crops/diagnostics (or old semantic results) with a new run.
    if output_dir.exists() and any(output_dir.iterdir()):
        raise FileExistsError(f"Output directory is not empty: {output_dir}. "
                              "Move or clear it before rerunning; no files have been deleted.")
    for name in ("dog_crops", "box_diagnostics"):
        (output_dir / name).mkdir(parents=True, exist_ok=True)
    return output_dir

#### 1.3.3 - Model Loading Functions



In [ ]:
def load_model(model_family, model_type, model_path=None):
    if model_family not in ("YOLO", "RFDETR"):
        raise ValueError("MODEL_FAMILY must be YOLO or RFDETR")
    if model_type not in ("baseline", "finetuned"):
        raise ValueError("MODEL_TYPE must be baseline or finetuned")
    if model_type == "finetuned" and not model_path:
        raise ValueError("A fine-tuned checkpoint path is required")
    if model_family == "YOLO":
        from ultralytics import YOLO
        return YOLO(str(model_path) if model_path else "yolo26n.pt")
    from rfdetr import RFDETRNano
    if model_type == "baseline":
        return RFDETRNano(pretrain_weights=str(model_path)) if model_path else RFDETRNano()
    return RFDETRNano.from_checkpoint(str(model_path))


def normalise_class_names(names):
    items = names.items() if isinstance(names, dict) else enumerate(names)
    return {int(i): str(name).strip().lower() for i, name in items}


def find_dog_class_id(names):
    matches = [i for i, name in normalise_class_names(names).items() if name == "dog"]
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one dog class in model metadata; got {matches}")
    return matches[0]


def get_dog_class_id(model, model_family, model_type, override=None):
    """None means use the fine-tuned RF-DETR detection-name metadata."""
    if override is not None:
        if int(override) != override or override < 0:
            raise ValueError("DOG_CLASS_ID_OVERRIDE must be a non-negative integer")
        return int(override)
    if model_family == "YOLO":
        return find_dog_class_id(model.names)
    if model_family == "RFDETR" and model_type == "baseline":
        from rfdetr.assets.coco_classes import COCO_CLASSES
        return find_dog_class_id(COCO_CLASSES)  # sparse COCO ID 18, not list index 16
    if model_family == "RFDETR" and model_type == "finetuned":
        return None  # use result.data["class_name"]; never use pretrained COCO IDs
    raise ValueError(f"Unsupported model: {model_family}/{model_type}")

#### 1.3.4 - Prediction

In [ ]:
def standardise_predictions(boxes, scores, accepted):
    """Return original-image xyxy boxes and confidences, in detector output order."""
    if not (len(boxes) == len(scores) == len(accepted)):
        raise ValueError("Detector returned inconsistent prediction lengths")
    return [{"box": box.tolist(), "confidence": float(score)}
            for box, score, keep in zip(np.asarray(boxes), scores, accepted) if keep]


def predict_yolo(model, image_path, dog_class_id, confidence):
    result = model.predict(source=str(image_path), classes=[dog_class_id],
                           conf=confidence, imgsz=YOLO_IMAGE_SIZE, verbose=False)[0]
    if result.boxes is None or len(result.boxes) == 0:
        return []
    boxes = result.boxes.xyxy.cpu().numpy()
    scores = result.boxes.conf.cpu().numpy()
    class_ids = result.boxes.cls.cpu().numpy()
    return standardise_predictions(boxes, scores, class_ids == dog_class_id)


def predict_rfdetr(model, image_path, dog_class_id, confidence):
    result = model.predict(images=str(image_path), threshold=confidence)
    if result is None or len(result) == 0:
        return []
    class_ids = np.asarray(result.class_id)
    if dog_class_id is not None:
        # Baseline: official sparse ID. Custom: explicitly verified override.
        accepted = class_ids == dog_class_id
    else:
        # Current RF-DETR versions resolve custom IDs using checkpoint metadata.
        names = getattr(result, "data", {}).get("class_name")
        if names is None:
            # Old versions may not provide class-name metadata. Prefer a real ID
            # mapping, and do not guess a custom ID from a list's position.
            mapping = getattr(model, "class_names", None)
            if not isinstance(mapping, dict):
                raise RuntimeError("RF-DETR custom prediction class names unavailable. "
                                   "Verify the checkpoint's dog prediction ID and set "
                                   "DOG_CLASS_ID_OVERRIDE (often 0 on current versions).")
            accepted = class_ids == find_dog_class_id(mapping)
        else:
            accepted = [str(name).strip().lower() == "dog" for name in names]
    return standardise_predictions(result.xyxy, result.confidence, accepted)


def detect_dogs(model, image_path, model_family, dog_class_id, confidence=MIN_PRED_CONF):
    if model_family == "YOLO":
        return predict_yolo(model, image_path, dog_class_id, confidence)
    if model_family == "RFDETR":
        return predict_rfdetr(model, image_path, dog_class_id, confidence)
    raise ValueError(f"Unsupported model family: {model_family}")

#### 1.3.5 - Actual vs Predicted Box Matching

In [ ]:
def load_ground_truth(label_path, image_width, image_height):
    if not label_path.exists():
        if MISSING_LABELS_ARE_NEGATIVE:
            return []
        raise FileNotFoundError(
            f"Missing validation label: {label_path}"
        )
    boxes = []
    lines = label_path.read_text().splitlines()
    for line in lines:
        if not line.strip():
            continue
        values = line.split()
        if len(values) != 5:
            raise ValueError(
                f"Invalid YOLO label in {label_path}: {line}"
            )
        class_id, xc, yc, bw, bh = map(float, values)
        # Our dataset should contain only class 0: dog
        if int(class_id) != 0:
            raise ValueError(
                f"Unexpected class {class_id} in {label_path}"
            )
        # Convert normalised centre coordinates
        # into original image pixel coordinates.
        x1 = (xc - bw / 2) * image_width
        y1 = (yc - bh / 2) * image_height
        x2 = (xc + bw / 2) * image_width
        y2 = (yc + bh / 2) * image_height
        # Clip coordinates to image boundaries
        x1 = max(0, min(image_width, x1))
        x2 = max(0, min(image_width, x2))
        y1 = max(0, min(image_height, y1))
        y2 = max(0, min(image_height, y2))
        if x2 > x1 and y2 > y1:
            boxes.append([x1, y1, x2, y2])
    return boxes


def calculate_iou(box_a, box_b):
    # Intersection coordinates
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])
    intersection_width = max(0, x2 - x1)
    intersection_height = max(0, y2 - y1)
    intersection_area = (
        intersection_width * intersection_height
    )
    # Individual box areas
    area_a = (
        (box_a[2] - box_a[0]) *
        (box_a[3] - box_a[1])
    )
    area_b = (
        (box_b[2] - box_b[0]) *
        (box_b[3] - box_b[1])
    )
    union_area = area_a + area_b - intersection_area
    if union_area <= 0:
        return 0.0
    return intersection_area / union_area


def match_boxes(predictions, ground_truth, iou_threshold=0.5):
    # Construct possible prediction / ground-truth matches
    pairs = []
    for pred_index, pred in enumerate(predictions):
        for gt_index, gt in enumerate(ground_truth):
            iou = calculate_iou(
                pred["box"],
                gt
            )
            pairs.append(
                (iou, pred_index, gt_index)
            )
    # Highest IoU matches first
    pairs.sort(reverse=True)
    matched_predictions = set()
    matched_ground_truth = set()
    # Each predicted box and ground-truth box
    # can only be matched once.
    for iou, pred_index, gt_index in pairs:
        if iou < iou_threshold:
            break
        if pred_index in matched_predictions:
            continue
        if gt_index in matched_ground_truth:
            continue
        matched_predictions.add(pred_index)
        matched_ground_truth.add(gt_index)
    return matched_predictions, matched_ground_truth


def safe_divide(a, b):
    return a / b if b else 0.0

## Section 2 - Dog Emotion Classifier

## Section 3 - Combining Both Models

### Section 3.1 - Testing Data Creation

### Section 3.2 - Dog Detection and Cropping in Testing